In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
cd /content/drive/MyDrive

/content/drive/MyDrive


In [3]:
ls

'Colab Notebooks'/   KDD2022@   __MACOSX/   taxi_drop/   taxi_drop.zip


# Data Preprocess & DataLoader

In [4]:
import numpy as np
import os
import torch
from torch.utils.data import TensorDataset, DataLoader

"""数据加载和预处理"""
# data["xxx_catagory"] = (num_samples, num_time_steps, locations, channels)
# eg: data["x_train"].shape: (2606, 12, 266, 3), data["y_train"].shape: (2606, 12, 266, 1)
# 论文： 输入：（P，N，C） P：历史时间步数，N：位置数量，C：通道数量
# channel： 0: traffic flow, 1: time 0-47/48 48個 , 2: day 0-6七天
# 使用dataloader直接加载数据，和st-llm不同，没有对最后剩余凑不满的数据进行填充，因此大小会不一样，最后一个batch会变小

# 数据标准化
class StandardScaler:
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std

    def transform(self, data): #标准化数据
        return (data - self.mean) / self.std

    def inverse_transform(self, data): #还原数据
        return (data * self.std) + self.mean

def load_dataset(dataset_dir, batch_size, valid_batch_size=None, test_batch_size=None):
    data = {}
    for category in ["train", "val", "test"]:
        cat_data = np.load(os.path.join(dataset_dir, category + ".npz"))
        data["x_" + category] = cat_data["x"]
        data["y_" + category] = cat_data["y"]
    #对第一维channel进行标准化
    scaler = StandardScaler(
        mean=data["x_train"][..., 0].mean(), std=data["x_train"][..., 0].std() # 为什么：only scale the first channel traffic flow
    )
    for category in ["train", "val", "test"]:
        data["x_" + category][..., 0] = scaler.transform(data["x_" + category][..., 0])

    train_ds = TensorDataset(torch.Tensor(data["x_train"]), torch.Tensor(data["y_train"]))
    val_ds   = TensorDataset(torch.Tensor(data["x_val"]),   torch.Tensor(data["y_val"]))

    data["train_loader"] = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    data["val_loader"] = DataLoader(val_ds, batch_size=valid_batch_size, shuffle=False)

    # test_ds  = TensorDataset(torch.Tensor(data["x_test"]),  torch.Tensor(data["y_test"]))
    # data["test_loader"] = DataLoader(test_ds, batch_size=test_batch_size, shuffle=False)
    
    data["scaler"] = scaler

    return data

In [18]:
"""测试dataLoader模块"""
dataset_dir = "taxi_drop"
taxi_drop_data = load_dataset(dataset_dir, batch_size=32, valid_batch_size=32, test_batch_size=32)

In [19]:
example_x, example_y = next(iter(taxi_drop_data["train_loader"]))
print("Example x shape:", example_x.shape)  # (batch_size, num_time_steps, locations, channels)
print("Example y shape:", example_y.shape)  # (batch_size, num_time_steps, locations, 1)

Example x shape: torch.Size([32, 12, 266, 3])
Example y shape: torch.Size([32, 12, 266, 1])


In [ ]:
# example_x[0,0,100,:]

# ST_LLM Model

In [9]:
import torch
import torch.nn as nn
from transformers.models.gpt2.modeling_gpt2 import GPT2Model

In [22]:
"""時間embedding:包含一天中的时间段和一周中的星期几信息"""
#輸入： x : (B, P, N, C) B: batch_size, P：历史时间步数，N：位置数量，C：通道数量
#輸出： time_embedding shape: (B, n_embd, N, 1)
class TemporalEmbedding(nn.Module):
    def __init__(self, time, n_embd):
        super().__init__()
        self.day = time
        self.time_day = nn.Embedding(time, n_embd)
        self.time_week = nn.Embedding(7, n_embd)
        nn.init.xavier_uniform_(self.time_day.weight)
        nn.init.xavier_uniform_(self.time_week.weight)

    # def forward(self, x): # x shape: (B, P, N, C)
    #     day_info = x[..., 1]  # (B, P, N)
    #     day_idx = (day_info[:, -1, :] * self.day).long().clamp(0, self.day - 1)  # (B, N)
    #     day_embd = self.time_day(day_idx)                      # (B, N, n_embd)
    #     day_embd = day_embd.transpose(1, 2).unsqueeze(-1)      # (B, n_embd, N, 1)

    #     week_info = x[..., 2]  # (B, P, N)
    #     week_idx = week_info[:, -1, :].long().clamp(0, 6)      # (B, N)
    #     week_embd = self.time_week(week_idx)                   # (B, N, n_embd)
    #     week_embd = week_embd.transpose(1, 2).unsqueeze(-1)    # (B, n_embd, N, 1)

    #     return day_embd + week_embd  # (B, n_embd, N, 1)
    

    def forward(self, x): # x shape: (B, P, N, C)
        # 获取输入数据所在的设备 (cuda:0 或 cpu)
        device = x.device 

        # --- 处理 Day Embedding ---
        day_info = x[..., 1]  # (B, P, N)
        # 显式使用 .to(device) 确保索引在 GPU 上
        day_idx = (day_info[:, -1, :] * self.day).long().clamp(0, self.day - 1).to(device)
        day_embd = self.time_day(day_idx)                      # (B, N, n_embd)
        day_embd = day_embd.transpose(1, 2).unsqueeze(-1)      # (B, n_embd, N, 1)

        # --- 处理 Week Embedding ---
        week_info = x[..., 2]  # (B, P, N)
        # 同样显式指定设备
        week_idx = week_info[:, -1, :].long().clamp(0, 6).to(device)
        week_embd = self.time_week(week_idx)                   # (B, N, n_embd)
        week_embd = week_embd.transpose(1, 2).unsqueeze(-1)    # (B, n_embd, N, 1)

        return day_embd + week_embd  # (B, n_embd, N, 1)


In [23]:
# """空間 embedding：每個地點學習一個獨立的向量"""
# # 輸入： x : (B, P, N, C)
# # 輸出： (B, n_embd, N, 1)
# class SpatialEmbedding(nn.Module):
#     def __init__(self, num_locations, n_embd):
#         super().__init__()
#         self.num_locations = num_locations
#         self.location_embd = nn.Embedding(num_locations, n_embd)  # table shape: (N, n_embd)
#         nn.init.xavier_uniform_(self.location_embd.weight)

#     def forward(self,x):  # x shape: (B, P, N, C)
#         B= x.shape[0]
#         node_emb = self.location_embd(torch.arange(self.num_locations))     # (N, n_embd)
#         node_emb = node_emb.unsqueeze(0).expand(B, -1, -1).transpose(1,2).unsqueeze(-1)  # (B, n_embd, N, 1)

#         return node_emb  # (B, n_embd, N, 1)
    
class SpatialEmbedding(nn.Module):
    def __init__(self, num_locations, n_embd):
        super().__init__()
        self.num_locations = num_locations
        self.location_embd = nn.Embedding(num_locations, n_embd)
        nn.init.xavier_uniform_(self.location_embd.weight)

    def forward(self, x):  # x shape: (B, P, N, C)
        B = x.shape[0]
        # 核心修改：显式指定 device
        device = x.device 
        indices = torch.arange(self.num_locations, device=device)
        
        node_emb = self.location_embd(indices) # (N, n_embd)
        node_emb = node_emb.unsqueeze(0).expand(B, -1, -1).transpose(1, 2).unsqueeze(-1)  # (B, n_embd, N, 1)

        return node_emb  # (B, n_embd, N, 1)


In [24]:
"""信息embedding：利用卷積融合流量+時間+空間信息"""
class TokenEmbedding(nn.Module):
    def __init__(self, input_dim = 3, input_len = 12, n_embd = 256):
        super().__init__()
        #卷積：（B，256，H，W)
        self.in_channels = input_dim * input_len # 3个通道，每个通道12个时间步，展开后是36维
        self.tk_embd = nn.Conv2d(in_channels=self.in_channels, out_channels=n_embd, kernel_size=1) #卷积核大小为1
        

    def forward(self, x): # x shape: (B, P, N, C)
        B, P, N, C = x.shape
        x = x.permute(0, 3, 1, 2) # (B, C, P, N)
        x = x.reshape(B, -1, N)    # (B, C*P, N)
        tk_embd = self.tk_embd(x.unsqueeze(-1)) # (B, n_embd, N, 1)

        return tk_embd # (B, n_embd, N, 1)

In [25]:
'''测试embedding模块'''
# # random_data = example_x # (batch_size, num_time_steps, locations, channels)
# token_embd = TokenEmbedding(input_dim=3, input_len=12, n_embd=256)
# temporal_embd = TemporalEmbedding(time=48, n_embd=256)
# spatial_embd = SpatialEmbedding(num_locations=266, n_embd=256)
# tk_embd = token_embd(example_x)
# print("Token Embedding shape:", tk_embd.shape)  
# sp_embd = spatial_embd(example_x)
# print("Spatial Embedding shape", sp_embd.shape)  # (B, n_embd, N, 1)
# tp_embd = temporal_embd(example_x)
# print("Temporal Embedding shape:", tp_embd.shape)  # (B, n_embd, N, 1)
# embd_feature = torch.cat([tk_embd,tp_embd,sp_embd], dim=1) # (B, 3*n_embd, N, 1)
# print(embd_feature.shape)
# fusion = nn.Conv2d(768,768,kernel_size=(1,1))
# data_feature = fusion(embd_feature)
# print(data_feature.shape)

'测试embedding模块'

In [26]:
'''部分冻结的大语言模型'''
#GPT2Block包含：
# - ln_1: LayerNorm
# - attn: GPT2Attention
# - ln_2: LayerNorm
# - mlp: GPT2MLP
# 原始一共12层
# GPT2輸入： batch, sequence_length, embedding_dim
# TODO:這個地方用序列建模合理嗎？
from transformers.models.gpt2.modeling_gpt2 import GPT2Model

class PFA(nn.Module):
    def __init__(self, gpt_layers = 6, U = 1): 
        super().__init__()
        self.gpt2 = GPT2Model.from_pretrained("gpt2")
        #保留中間的attention權重：self.gpt2 = GPT2Model.from_pretrained("gpt2", output_attentions=True, output_hidden_states=True）
        self.gpt2.h = self.gpt2.h[:gpt_layers] #保留前gpt_layers层，注意原始的gpt2是12層
        self.U = U #每个位置的邻居数量

        # frooze 和 learning 的 param调整
        for layer_idx, layer in enumerate(self.gpt2.h):
            for name, param in layer.named_parameters():
                if layer_idx < gpt_layers - self.U:  #所有層數 - 不凍結attention的層數,前幾層
                    #TODO: wpe是什么
                    if "ln" in name: # LayerNorm层
                        param.requires_grad = True
                    else:
                        param.requires_grad = False
                else: #最後U層，attn也要打開
                    if "mlp" in name:
                        param.requires_grad = False
                    else:
                        param.requires_grad = True

    def forward(self, x): # x shape: (batch_size, num_time_steps, locations, n_embd)
        return self.gpt2(inputs_embeds = x).last_hidden_state

In [27]:
"""測繪pfa模塊"""
# in_data = torch.randn(32, 266, 768) # (B, N, gpt_embd)
# pfa = PFA(gpt_layers=6, U=1)
# out_data = pfa(in_data)
# print("PFA output shape:", out_data.shape)  # (B, N, gpt_embd)

'測繪pfa模塊'

In [28]:
class ST_LLM(nn.Module):
    def __init__(self, batch_size = 32, num_nodes = 266, input_len = 12, output_len = 12, gpt_layers = 6, U = 1):
        super().__init__()
        self.batch_size = batch_size
        self.num_nodes = num_nodes #location数量
        self.input_len = input_len
        self.output_len = output_len
        self.gpt_layers = gpt_layers
        self.U = U
        self.time = 48      #一天中的时间段数量
        self.n_embd = 256   #时间，空间，token三个拼一起，3*256=768
        self.gpt_embd = 768 #embedding维度，和GPT-2的默认维度一致
        #embedding
        self.time_embd = TemporalEmbedding(self.time, self.n_embd) #时间编码模块
        self.spatial_embd = SpatialEmbedding(self.num_nodes, self.n_embd) #空间编码模块
        self.token_embd = TokenEmbedding(input_dim=3, input_len=self.input_len, n_embd=self.n_embd) #token embedding, 将原始的3个通道映射到n_embd维度
        #聚合embedding
        self.feature_fuse = nn.Conv2d(in_channels=3*self.n_embd, out_channels=self.gpt_embd, kernel_size=1) #特征融合，卷积核大小为1，输入（B，xxx，H，W）
        #PFA
        self.pfa = PFA(gpt_layers=self.gpt_layers, U=self.U)
        #最后解码成预测结果
        self.regression_layer = nn.Conv2d(in_channels=self.gpt_embd, out_channels=self.output_len, kernel_size=1) #卷积核大小为1，输入（B，gpt_embd，H，W）
        

    def forward(self, history_data):    # x shape: (B, P, N, C)
        input_data = history_data
        B, P, N, C = input_data.shape

        #embedding
        time_embd = self.time_embd(input_data) # (B, n_embd, N, 1)
        spatial_embd = self.spatial_embd(input_data) # (B, n_embd, N, 1)
        token_embd = self.token_embd(input_data) # (B, n_embd, N, 1)

        #feature fusing
        embd_feature = torch.cat([token_embd, time_embd, spatial_embd], dim=1) # (B, 3*n_embd, N, 1)
        data_feature = self.feature_fuse(embd_feature) # (B, gpt_embd,N,1)
        data_feature = data_feature.squeeze(-1).permute(0, 2, 1) # (B, N, gpt_embd)

        #PFA
        pfa_out = self.pfa(data_feature) # (B, N, gpt_embd)

        #最終預測
        pfa_out = pfa_out.permute(0, 2, 1).unsqueeze(-1) # (B, gpt_embd, N, 1)

        pred = self.regression_layer(pfa_out) # (B, output_len, N, 1)
        pred = pred.squeeze(-1) # (B, output_len, N)
        
        return pred

In [20]:
"""测试ST_LLM模型"""
model = ST_LLM(batch_size=32, num_nodes=266, input_len=12, output_len=12, gpt_layers=6, U=1)
pred = model(example_x) # (B, output_len, N, 1)
print("Prediction shape:", pred.shape)  # (B, output_len, N, 1)
scaler = taxi_drop_data['scaler']
prediction = scaler.inverse_transform(pred) #还原到原始尺度
print("Inverse transformed prediction shape:", prediction.shape)  # (B, output_len, N)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2Model LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Prediction shape: torch.Size([32, 12, 266])
Inverse transformed prediction shape: torch.Size([32, 12, 266])


# Train

In [29]:
# train 
import torch
import numpy as np
import pandas as pd

torch.manual_seed(313)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") 
print("Using Device:{device}",device)

#TODO:自动化参数输入
lrate = 1e-3
wdecay = 1e-4
epochs = 300
num_nodes = 266
input_len = 12
output_len = 12
gpt_layers = 6
U = 1

"""训练循环和评估指标计算"""
class trainer:
    def __init__(self,scaler, learning_rate, wdecay,
                 num_nodes, input_len, output_len, gpt_layers, U):
        # scaler 是 dataloader["scaler"]
        self.scaler = scaler
        self.model = ST_LLM(batch_size=32, num_nodes=num_nodes, 
                            input_len=input_len, output_len=output_len,
                            gpt_layers=gpt_layers, U=U).to(device) #model to device
        self.optimizer = torch.optim.AdamW(
                filter(lambda p: p.requires_grad, self.model.parameters()), 
                lr=learning_rate, 
                weight_decay=wdecay)
        self.criterion = nn.MSELoss()

    #输入一个batch的数据
    def train_loop(self, x, y):# x:(B, P, N, C), y:(B, P, N, 1)
        model = self.model
        model.train()
        self.optimizer.zero_grad()
        output = model(x)   #output: (B, output_len, N, 1)
        pred = self.scaler.inverse_transform(output).unsqueeze(-1) #还原到原始尺度
        loss = self.criterion(pred, y)
        loss.backward()
        self.optimizer.step()
        return loss.item()
    
    def eval_loop(self, x, y):
        model = self.model
        model.eval()
        with torch.no_grad():
            output = model(x)  #output: (B, output_len, N, 1)
            pred = self.scaler.inverse_transform(output).unsqueeze(-1) #还原到原始尺度
            loss = self.criterion(pred, y)
        return loss.item()
    
def main():
    # 这里可以设置训练参数，加载数据，创建trainer实例，并调用train_loop进行训练
    #TODO:参数输入
    history_loss = []
    
    dataset_dir = "taxi_drop"
    dataloader = load_dataset(dataset_dir, batch_size=32, valid_batch_size=32, test_batch_size=32)
    engine = trainer(scaler=dataloader["scaler"], learning_rate=lrate, wdecay=wdecay,
                     num_nodes=num_nodes, input_len=input_len, output_len=output_len,
                     gpt_layers=gpt_layers, U=U)
    for epoch in range(epochs):
        train_loss = []
        print(f"Epoch {epoch+1}/{epochs}")
        for x_batch, y_batch in dataloader["train_loader"]:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            train_metrics = engine.train_loop(x_batch, y_batch)
            train_loss.append(train_metrics)

        valid_loss = []
        for x_batch, y_batch in dataloader["val_loader"]:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            valid_metrics = engine.eval_loop(x_batch, y_batch)
            valid_loss.append(valid_metrics)
        
        print(f"Epoch {epoch+1} Train Loss: {np.mean(train_loss):.4f}, Valid Loss: {np.mean(valid_loss):.4f}")

if __name__ == "__main__":
    main()


Using Device:{device} cuda


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2Model LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/300
Epoch 1 Train Loss: 938.9679, Valid Loss: 419.1381
Epoch 2/300
Epoch 2 Train Loss: 377.5264, Valid Loss: 226.0309
Epoch 3/300
Epoch 3 Train Loss: 233.9827, Valid Loss: 221.5144
Epoch 4/300
Epoch 4 Train Loss: 179.7951, Valid Loss: 161.7083
Epoch 5/300
Epoch 5 Train Loss: 154.4856, Valid Loss: 152.0005
Epoch 6/300
Epoch 6 Train Loss: 136.0543, Valid Loss: 148.7341
Epoch 7/300
Epoch 7 Train Loss: 125.4866, Valid Loss: 149.2821
Epoch 8/300
Epoch 8 Train Loss: 120.3052, Valid Loss: 149.8356
Epoch 9/300
Epoch 9 Train Loss: 113.6224, Valid Loss: 143.1980
Epoch 10/300
Epoch 10 Train Loss: 108.5783, Valid Loss: 129.6414
Epoch 11/300
Epoch 11 Train Loss: 104.1277, Valid Loss: 148.0840
Epoch 12/300
Epoch 12 Train Loss: 102.3501, Valid Loss: 129.2844
Epoch 13/300
Epoch 13 Train Loss: 98.4366, Valid Loss: 125.3956
Epoch 14/300
Epoch 14 Train Loss: 97.1657, Valid Loss: 133.6016
Epoch 15/300
Epoch 15 Train Loss: 94.6489, Valid Loss: 127.9433
Epoch 16/300
Epoch 16 Train Loss: 94.9262, Val